# UNet Segmentation Pipeline

Main orchestration notebook for the U-Net segmentation pipeline.
Run each stage independently as needed:

| Stage | When to run |
|---|---|
| 1 – Preprocessing | Once per dataset (or when data changes) |
| 2 – Tuning | Optional; skip if using default hyperparameters |
| 3 – Training | Each time you want to train a new model |
| 4 – Evaluation | After training, to measure final performance |

In [1]:
# Configuration — choose which config file to use.
# Each config file contains all ML hyperparameters (learning rate, batch size,
# model architecture, etc.). Swap to a different config by changing this import.
import config.configUnetxAE as configuration

# Pipeline stage modules
import preprocessing
import tuning
import training
import evaluation

# Load and validate the configuration.
# validate() checks that all required parameters are present and sensible.
config = configuration.Configuration().validate()
print("Configuration loaded:", config.modality)

Configuration loaded: AE


## Setup

## Stage 1: Preprocessing

Loads raw geospatial images, normalizes pixel values, creates image chips, and splits into train/val/test sets.

**Run when:** You have new or modified input data. Skip if you have already preprocessed and chips are saved to disk.

In [2]:
preprocessing.preprocess_all(config)

Starting preprocessing.
Reading training data shapefiles.. Done in 0.21 seconds. Found 10794 polygons in 39 areas.
Assigning polygons to areas..      Done in 0.06 seconds.
Assigning areas to input images..  Done in 0.00 seconds. Found 10 training images of which 10 contain training areas.



Extracting areas for Goldstream_10152025_rgb_MavicPro_WholeLake_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 9799.78it/s]
Extracting areas for Goldstream_10192025_rgb_MavicPro_WholeLake_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 6310.39it/s]
Extracting areas for Goldstream_10202025_rgb_MavicPro_WholeLake_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 10875.46it/s]
Extracting areas for Goldstream_10212025_rgb_MavicPro_WholeLake_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 9769.34it/s]
Extracting areas for Goldstream_10232025_rgb_MavicPro_WholeLake_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 12064.15it/s]
Extracting areas for Octopus_10152025_rgb_MavicPro_South_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 13012.32it/s]
Extracting areas for Octopus_10192025_rgb_MavicPro_WholeLake_post_cog_rectified.tif: 100%|██████████| 3/3 [00:00<00:00, 13443.28it/s]
Extracting areas for Octopus_10202025_rgb_MavicPro_Wh

[SPLIT][DATA] Reading train/val/test split from file
[SPLIT][DATA] training_frames=23
[SPLIT][DATA] validation_frames=8
[SPLIT][DATA] testing_frames=8

[DATA][STATS] positive-rate % by frame - mean | median | std | min..max  (n)
  train: 2.289 | 1.250 | 3.078 | 0.000..13.101  (n=23)
    val: 1.557 | 1.053 | 1.614 | 0.000..5.114  (n=8)
   test: 1.751 | 0.492 | 2.382 | 0.047..7.217  (n=8)


Building chips (no-overlap CC): 100%|██████████| 39/39 [00:28<00:00,  1.38it/s]

Preprocessing completed in 0:00:30.



## Stage 2: Hyperparameter Tuning (Optional)

Automatically searches for the best hyperparameters (learning rate, dropout, etc.) using Optuna.

**Run when:** Starting fresh or when default parameters are not yielding good results. This is computationally expensive — skip if you have a known good config.

In [ ]:
# Uncomment to run hyperparameter tuning:
# best = tuning.tune_UNet(config)
# print("Best hyperparameters:", best)

## Stage 3: Training

Trains the U-Net model. Running multiple iterations (the loop below) averages results across random seeds, giving more stable performance estimates.

### TensorBoard
Run the cell below to open TensorBoard inline. It will show live loss curves and image predictions as training progresses.

> **Note:** If TensorBoard does not appear inline, open a terminal and run:
> ```
> venv-bubble/Scripts/tensorboard --logdir C:\Users\amwelch3\git_repos\bubble-mapping/data/logs/UNET/AE
> ```
> then open `http://localhost:6006` in your browser.

In [ ]:
# Launch TensorBoard inline.
# Logs are written to config.logs_dir during training.
%load_ext tensorboard
%tensorboard --logdir {config.logs_dir}

In [3]:
# Train the model. Each iteration starts from fresh random weights.
# Multiple iterations reduce variance in the final metrics.
for i in range(10):
    print(f"\n =========== Starting training iteration {i+1}/10 ===========\n")
    training.train_UNet(config)


 =========== Starting training iteration 1/10 ===========

Starting training (UNet).
[DATA][LOAD] Found 39 input frames in C:\Users\amwelch3\git_repos\bubble-mapping/data/preprocessed/2026-03-26_UNETxAE


Processing frames: 100%|██████████| 39/39 [00:03<00:00, 11.88it/s]


[SPLIT][DATA] Reading train/val/test split from file
[SPLIT][DATA] training_frames=23
[SPLIT][DATA] validation_frames=8
[SPLIT][DATA] testing_frames=8

[DATA][STATS] positive-rate % by frame - mean | median | std | min..max  (n)
  train: 2.289 | 1.250 | 3.078 | 0.000..13.101  (n=23)
    val: 1.557 | 1.053 | 1.614 | 0.000..5.114  (n=8)
   test: 1.751 | 0.492 | 2.382 | 0.047..7.217  (n=8)

[UNET][TRAIN]  run=UNETxAE  model_name=UNETxAE
[UNET][TRAIN]  device=cuda, amp_dtype=bf16, channels_last=True
[UNET][TRAIN]  ema=False (decay=0.999)
[UNET][TRAIN]  epochs=10, steps/epoch=50, val_steps=50, batch=8, workers=0
[UNET][TRAIN]  patch=(256, 256), stride=None, aug=0.7, min_pos_frac=0.001, pos_ratio=None
[UNET][TRAIN]  loss=tversky (alpha=0.55, beta=0.45), optimizer=adamw, scheduler=onecycle, lr=4.00e-04
[UNET][TRAIN]  logs_dir=C:\Users\amwelch3\git_repos\bubble-mapping/data/logs/UNET/AE\20260326-0950_UNETxAE_unet



New best! val_loss improved inf -> 0.961395. Saved: C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-0950_UNETxAE.weights.pt

 Epoch 1/10 [00:00:40]  lr=1.13e-04
  train: loss=0.9695 | dice_coef=0.0324 | accuracy=0.4748
   val: val_loss=0.9614 | val_dice_coef=0.0412 | val_accuracy=0.0290 | val_specificity=0.0042 | val_sensitivity=0.9957 | val_f_beta=0.0872 | val_f1_score=0.0433 | val_IoU=0.0246 | val_normalized_surface_distance=0.0422 | val_Hausdorff_distance=285.3119 | val_boundary_intersection_over_union=0.0131 | val_dice_loss=0.9567 | best_val_loss=0.9614


New best! val_loss improved 0.961395 -> 0.956870. Saved: C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-0950_UNETxAE.weights.pt

 Epoch 2/10 [00:00:37]  lr=3.06e-04
  train: loss=0.9545 | dice_coef=0.0479 | accuracy=0.3851
   val: val_loss=0.9569 | val_dice_coef=0.0466 | val_accuracy=0.0332 | val_specificity=0.0065 | val_sensitivity=0.9995 | val_f_beta=0.0770 | val_f1_score=0.0382 | val_IoU=0.0210 | val_normalized_surface_distance=0.0213 | val_Hausdorff_distance=269.6538 | val_boundary_intersection_over_union=0.0101 | val_dice_loss=0.9618 | best_val_loss=0.9569


...

New best! val_loss improved 0.952739 -> 0.950235. Saved: C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-1047_UNETxAE.weights.pt

 Epoch 6/10 [00:00:38]  lr=2.43e-04
  train: loss=0.9264 | dice_coef=0.0750 | accuracy=0.5190
   val: val_loss=0.9502 | val_dice_coef=0.0536 | val_accuracy=0.0523 | val_specificity=0.0220 | val_sensitivity=0.9834 | val_f_beta=0.1095 | val_f1_score=0.0569 | val_IoU=0.0320 | val_normalized_surface_distance=0.0438 | val_Hausdorff_distance=261.6200 | val_boundary_intersection_over_union=0.0204 | val_dice_loss=0.9431 | best_val_loss=0.9502


New best! val_loss improved 0.950235 -> 0.828338. Saved: C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-1047_UNETxAE.weights.pt

 Epoch 7/10 [00:00:38]  lr=1.54e-04
  train: loss=0.9202 | dice_coef=0.0807 | accuracy=0.4934
   val: val_loss=0.8283 | val_dice_coef=0.1712 | val_accuracy=0.8892 | val_specificity=0.9101 | val_sensitivity=0.6263 | val_f_beta=0.3242 | val_f1_score=0.1308 | val_IoU=0.0966 | val_normalized_surface_distance=0.2027 | val_Hausdorff_distance=216.5784 | val_boundary_intersection_over_union=0.1816 | val_dice_loss=0.8692 | best_val_loss=0.8283



 Epoch 8/10 [00:00:38]  lr=7.39e-05
  train: loss=0.8986 | dice_coef=0.1017 | accuracy=0.5752
   val: val_loss=0.8408 | val_dice_coef=0.1544 | val_accuracy=0.9525 | val_specificity=0.9746 | val_sensitivity=0.5559 | val_f_beta=0.3934 | val_f1_score=0.1572 | val_IoU=0.1155 | val_normalized_surface_distance=0.2985 | val_Hausdorff_distance=174.7510 | val_boundary_intersection_over_union=0.2725 | val_dice_loss=0.8428 | best_val_loss=0.8283



 Epoch 9/10 [00:00:38]  lr=1.90e-05
  train: loss=0.8981 | dice_coef=0.1035 | accuracy=0.4971
   val: val_loss=0.9532 | val_dice_coef=0.0441 | val_accuracy=0.9642 | val_specificity=0.9841 | val_sensitivity=0.4188 | val_f_beta=0.2739 | val_f1_score=0.0502 | val_IoU=0.0311 | val_normalized_surface_distance=0.2464 | val_Hausdorff_distance=221.4585 | val_boundary_intersection_over_union=0.2395 | val_dice_loss=0.9498 | best_val_loss=0.8283



 Epoch 10/10 [00:00:38]  lr=9.66e-09
  train: loss=0.8797 | dice_coef=0.1220 | accuracy=0.5331
   val: val_loss=0.9522 | val_dice_coef=0.0450 | val_accuracy=0.9594 | val_specificity=0.9840 | val_sensitivity=0.4282 | val_f_beta=0.2621 | val_f1_score=0.0975 | val_IoU=0.0650 | val_normalized_surface_distance=0.2244 | val_Hausdorff_distance=209.6693 | val_boundary_intersection_over_union=0.1974 | val_dice_loss=0.9025 | best_val_loss=0.8283
Saved final model to: C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-1047_UNETxAE.pt
Training completed.

Training completed in 0:06:29.



## Stage 4: Evaluation

Evaluates the trained model on the held-out test set. This is the only unbiased measure of real-world performance, because:
- Training data was used to learn weights
- Validation data was used to select the best checkpoint
- **Test data is completely unseen** — metrics here reflect true generalization

In [2]:
evaluation.evaluate_unet(config)

Starting evaluation for architecture:unet
MAKE SURE TO USE FULL AREAS FROM PREPROCESSED DIR
CURRENTLY USING THIS PATH AS INPUT: C:\Users\amwelch3\git_repos\bubble-mapping/data/preprocessed/2026-03-26_UNETxAE

[DATA][LOAD] Found 39 input frames in C:\Users\amwelch3\git_repos\bubble-mapping/data/preprocessed/2026-03-26_UNETxAE


Processing frames: 100%|██████████| 39/39 [00:03<00:00, 12.79it/s]


Reading train-test split from file
[SPLIT][DATA] Reading train/val/test split from file
[SPLIT][DATA] training_frames=23
[SPLIT][DATA] validation_frames=8
[SPLIT][DATA] testing_frames=8
training_frames 23
validation_frames 8
testing_frames 8

Testing frames: 8
Found 20 checkpoint(s).


[EVAL] model=unet  ckpt=C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-0950_UNETxAE.weights.pt
[EVAL] saving masks -> C:\Users\amwelch3\git_repos\bubble-mappingdata/results/UNET/AE\20260326-0950_UNETxAE.weights


Predicting (unet): 100%|██████████| 8/8 [01:55<00:00, 14.42s/it] 



[EVAL] model=unet  ckpt=C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-0950_UNETxAE.pt
[EVAL] saving masks -> C:\Users\amwelch3\git_repos\bubble-mappingdata/results/UNET/AE\20260326-0950_UNETxAE


...


[EVAL] model=unet  ckpt=C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-1047_UNETxAE.weights.pt
[EVAL] saving masks -> C:\Users\amwelch3\git_repos\bubble-mappingdata/results/UNET/AE\20260326-1047_UNETxAE.weights


Predicting (unet): 100%|██████████| 8/8 [01:40<00:00, 12.52s/it]



[EVAL] model=unet  ckpt=C:\Users\amwelch3\git_repos\bubble-mapping/data/models/UNET/AE\20260326-1047_UNETxAE.pt
[EVAL] saving masks -> C:\Users\amwelch3\git_repos\bubble-mappingdata/results/UNET/AE\20260326-1047_UNETxAE


Predicting (unet): 100%|██████████| 8/8 [01:39<00:00, 12.41s/it]


Wrote 20 result rows to C:\Users\amwelch3\git_repos\bubble-mappingdata/results/UNET/AE\evaluation_unet.csv

EVALUATION SUMMARY (UNET)

Checkpoint                                         Dice       IoU        Acc        Hausd      F1         Time      
--------------------------------------------------------------------------------------------------------------
20260326-0950_UNETxAE.weights.pt                   0.028728   0.014573   0.014807   950.128815 0.028728   00:12:50  
20260326-0950_UNETxAE.weights.pt                   0.028733   0.014576   0.015121   937.305548 0.028733   00:01:55  
20260326-0950_UNETxAE.pt                           0.215949   0.121044   0.985778   791.373169 0.215949   00:01:44  
20260326-0957_UNETxAE.weights.pt                   0.038149   0.019445   0.307486   734.515836 0.038149   00:01:40  
20260326-0957_UNETxAE.pt                           0.331050   0.198358   0.986536   743.667376 0.331050   00:01:39  
20260326-1003_UNETxAE.weights.pt                   